# Speaking Practice for the TOEFL Exam

**AI-Powered TOEFL Speaking Prep — Unlimited Practice**

This notebook runs the full TOEFL Speaking Practice platform with:
- **Listen & Repeat** — 50 practice sets
- **Take an Interview** — 50 interview topics
- **AI Evaluation** powered by Qwen2.5-3B-Instruct (runs locally on GPU)
- **Whisper** for speech transcription
- **Wav2Vec2** for pronunciation analysis
- **Google Drive** for progress persistence across sessions

---

## Instructions
1. Make sure the runtime is set to **GPU** (Runtime > Change runtime type > T4 GPU)
2. Run each cell in order (or use Runtime > Run all)
3. Allow Google Drive access when prompted
4. Upload the zip file when prompted
5. Click the public URL at the end to open the app in your browser
6. Allow microphone access when prompted

> **Note:** First run takes 5-10 minutes to download AI models. Subsequent runs are faster. Your progress is saved to Google Drive automatically.

## Step 1: Install Dependencies

In [ ]:
!pip install -q flask faster-whisper transformers accelerate g2p-en soundfile scipy bitsandbytes
!npm install -g localtunnel > /dev/null 2>&1

## Step 2: Mount Google Drive, Upload & Extract Project Files

Your progress is saved to Google Drive so it persists across sessions. Upload the `English_Speaking_Practice.zip` file when prompted.

In [ ]:
import os
import zipfile
import json
from google.colab import files as colab_files
from google.colab import drive

# Mount Google Drive to persist progress across sessions
drive.mount('/content/drive')
GDRIVE_DIR = '/content/drive/MyDrive/EnglishSpeakingPractice'
os.makedirs(GDRIVE_DIR, exist_ok=True)
GDRIVE_PROGRESS = os.path.join(GDRIVE_DIR, 'progress.json')

# Initialize progress file on Drive if it doesn't exist
if not os.path.exists(GDRIVE_PROGRESS):
    with open(GDRIVE_PROGRESS, 'w') as f:
        json.dump({'lr_completed': [], 'int_completed': []}, f)
    print('Created new progress file on Google Drive.')
else:
    with open(GDRIVE_PROGRESS, 'r') as f:
        p = json.load(f)
    print(f'Loaded existing progress from Google Drive: {len(p.get("lr_completed", []))} LR sets, {len(p.get("int_completed", []))} interviews completed.')

# Set env var so server.py picks up the Drive path
os.environ['GDRIVE_PROGRESS_PATH'] = GDRIVE_PROGRESS

print('Please upload English_Speaking_Practice.zip')
print()
uploaded = colab_files.upload()

zip_name = list(uploaded.keys())[0]
print(f'\nExtracting {zip_name}...')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/app')

# Handle case where zip contains a top-level folder
app_dir = '/content/app'
entries = os.listdir(app_dir)
if len(entries) == 1 and os.path.isdir(os.path.join(app_dir, entries[0])):
    import shutil
    subfolder = os.path.join(app_dir, entries[0])
    for item in os.listdir(subfolder):
        shutil.move(os.path.join(subfolder, item), os.path.join(app_dir, item))
    os.rmdir(subfolder)

print('\nExtracted files:')
for root, dirs, files_list in os.walk(app_dir):
    for fname in files_list:
        path = os.path.join(root, fname)
        print(f'  {os.path.relpath(path, app_dir)}')

print(f'\nAll files ready! Progress saved to Google Drive.')

## Step 3: Load AI Models

This downloads and loads Whisper, Wav2Vec2, and Qwen2.5-3B-Instruct. First run takes a few minutes.

In [ ]:
import sys
import torch
sys.path.insert(0, '/content/app')

print('Loading Whisper...')
from faster_whisper import WhisperModel
whisper = WhisperModel('medium', device='cuda', compute_type='float16')
print('Whisper loaded.')

print('\nLoading Wav2Vec2...')
import phoneme_analyzer
phoneme_analyzer.load_model()
print('Wav2Vec2 loaded.')

print('\nLoading Qwen2.5-3B-Instruct 4-bit...')
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
model_name = 'Qwen/Qwen2.5-3B-Instruct'
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map='auto',
)
print('Qwen2.5-3B-Instruct loaded.')

print('\nAll models loaded successfully!')

## Step 4: Start the App

This starts the server and gives you a public URL to access the app.

In [ ]:
import threading
import subprocess
import time
import logging
import urllib.request

os.chdir('/content/app')

# Patch server.py to use pre-loaded models
import server
server.whisper_model = whisper
server._phoneme_analyzer = phoneme_analyzer
server._llm_model = llm
server._llm_tokenizer = tokenizer

# Enable Flask/werkzeug request logging (like running locally in terminal)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%H:%M:%S')
werkzeug_log = logging.getLogger('werkzeug')
werkzeug_log.setLevel(logging.INFO)
server.app.logger.setLevel(logging.INFO)

# Start Flask in a background thread
def run_flask():
    server.app.run(port=5000, debug=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(2)

# Start localtunnel
lt_process = subprocess.Popen(
    ['lt', '--port', '5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)
time.sleep(3)
url_line = lt_process.stdout.readline().strip()

# Get the tunnel password (your public IP)
try:
    tunnel_password = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
except:
    tunnel_password = 'check your IP'

print(f'\n{"=" * 50}')
print(f'Your app is live!')
print(f'URL: {url_line}')
print(f'{"=" * 50}')
print(f'\nIf asked for a tunnel password, enter: {tunnel_password}')
print(f'\nKeep this notebook running while you practice.')
print(f'Close this tab or restart the runtime to stop.')